extract_label Function Test

In [ ]:
def extract_label(uri: str)->str:
    if "#" in uri:
        label = uri.split("#")[-1]
    else:
        label = uri.split("/")[-1]

    #replace function doesn't need exception handling if said item wasn't found to be replaced
    return label.replace("_", " ")

In [ ]:
#Case 1 with '#'
uri = "https://w3id.org/cskg/ontology#analyzesTask"
extract_label(uri)	

'analyzesTask'

In [ ]:
#Case 2 with '/'
uri = "https://w3id.org/cskg/resource/image_classification"
extract_label(uri)	

'image classification'

predicate_to_text Function Test


In [ ]:
PREDICATE_MAP = {
    "analyzesTask":        "analyzes the task of",
    "usesMethod":          "uses the method",
    "usesTask":            "uses the task of",
    "usesMaterial":        "uses the material",
    "producesMethod":      "produces the method",
    "executesMethod":      "executes the method",
    "matchesMethod":       "matches the method",
    "based-onMethod":      "is based on the method",
    "based-onTask":        "is based on the task of",
    "broader":             "is a broader concept than", 
    "narrower":            "is a narrower concept than",
    "related":             "is related to",
    "exactMatch":          "exactly matches",
    "closeMatch":          "closely matches",
    "hasMethod":           "has the method",
    "hasTask":             "has the task of",
    "hasMaterial":         "has the material",
}

def predicate_to_text(pred_label: str) -> str:

    #get method uses (key, default value), key is what its looking for, default value is what to return in case key is not found
    return PREDICATE_MAP.get(pred_label, pred_label)

In [24]:
#Case 1 with 'analyzesTask'
pred_label = "analyzesTask"
predicate_to_text(pred_label)

'analyzes the task of'

In [30]:
#Case 2 with 'usesMethod'
pred_label = "usesMethod"
predicate_to_text(pred_label)

'uses the method'

binding_to_sentence Function Test

In [ ]:
from urllib.parse import unquote

def binding_to_sentence(binding: dict) -> str:
   
    subject   = extract_label(binding["s"]["value"])
    predicate = predicate_to_text(extract_label(binding["p"]["value"]))
    obj       = extract_label(binding["o"]["value"])

    # DOI is URL-encoded in the data (e.g. "https%3A//doi.org/...")
    # unquote() converts it back to a normal URL
    doi_raw   = binding["doi"]["value"]
    doi       = unquote(doi_raw)
    support   = binding["support"]["value"]
    
    sentence = (
        f"{subject} {predicate} {obj} "
        f"(supported by {support} papers, source: {doi})"
    )
    return sentence

In [40]:
#Trial 1
binding = { 
    "s": { "type": "uri", "value": "https://w3id.org/cskg/resource/image_classification" }	,
    "p": { "type": "uri", "value": "https://w3id.org/cskg/ontology#analyzesTask" }	,
    "o": { "type": "uri", "value": "https://w3id.org/cskg/resource/task" }	,
    "doi": { "type": "typed-literal", "datatype": "http://www.w3.org/2001/XMLSchema#anyURI", "value": "https%3A//doi.org/10.1109/cvpr42600.2020.00378" }	,
    "support": { "type": "typed-literal", "datatype": "http://www.w3.org/2001/XMLSchema#integer", "value": "6" }
    }

binding_to_sentence(binding)

'image classification analyzes the task of task (supported by 6 papers, source: https://doi.org/10.1109/cvpr42600.2020.00378)'

In [41]:
#Trial 2
binding = { 
    "s": { "type": "uri", "value": "https://w3id.org/cskg/resource/image_classification" }	,
    "p": { "type": "uri", "value": "https://w3id.org/cskg/ontology#executesMethod" }	,
    "o": { "type": "uri", "value": "https://w3id.org/cskg/resource/neural_network" }	,
    "doi": { "type": "typed-literal", "datatype": "http://www.w3.org/2001/XMLSchema#anyURI", "value": "https%3A//doi.org/10.1145/3354031.3354045" }	, 
    "support": { "type": "typed-literal", "datatype": "http://www.w3.org/2001/XMLSchema#integer", "value": "11" }
    }

binding_to_sentence(binding)

'image classification executes the method neural network (supported by 11 papers, source: https://doi.org/10.1145/3354031.3354045)'